# 05b 동작 확인용 테스트

05b_reviews_individual.ipynb와 로직 동일. 차이점만 아래에 명시.

| | 테스트 | 본수집 |
|--|--------|--------|
| 입력 | 3개 게임 하드코딩 | `target_games.csv` 50개 |
| 게임당 최대 | 300개 (3페이지) | 50,000개 |
| 출력 | `review_individual_test.csv` | `review_individual.csv` |

In [1]:
import requests
import pandas as pd
import time
import os

TEST_GAMES = pd.DataFrame([
    {"appid": 105600,  "name": "Terraria",      "genre_category": "Action"},
    {"appid": 413150,  "name": "Stardew Valley", "genre_category": "Casual/Lightweight"},
    {"appid": 292030,  "name": "The Witcher 3",  "genre_category": "RPG"},
])

print(f"수집 대상: {len(TEST_GAMES)}개 게임")
print(TEST_GAMES[["appid", "name", "genre_category"]])

수집 대상: 3개 게임
    appid            name      genre_category
0  105600        Terraria              Action
1  413150  Stardew Valley  Casual/Lightweight
2  292030   The Witcher 3                 RPG


In [2]:
import requests
import pandas as pd
import time
import os
from datetime import datetime, timezone, timedelta

SLEEP_SEC            = 1.0
MAX_REVIEWS_PER_GAME = 300
MAX_RETRIES          = 3
OUTPUT_PATH          = "../data/review_individual_test.csv"

SINCE_DT = datetime.now(timezone.utc) - timedelta(days=3 * 365)
SINCE_TS = int(SINCE_DT.timestamp())

print(f"sleep: {SLEEP_SEC}초 / 게임당 최대: {MAX_REVIEWS_PER_GAME:,}개 / 재시도: {MAX_RETRIES}회")
print(f"수집 기간: {SINCE_DT.strftime('%Y-%m-%d')} 이후")
print(f"출력: {OUTPUT_PATH}")

sleep: 1.0초 / 게임당 최대: 300개 / 재시도: 3회
수집 기간: 2023-05-19 이후
출력: ../data/review_individual_test.csv


In [3]:
def collect_game_reviews(appid, max_reviews=MAX_REVIEWS_PER_GAME):
    collected = []
    cursor = "*"

    while len(collected) < max_reviews:
        params = {
            "json": 1,
            "filter": "recent",
            "language": "all",
            "review_type": "all",
            "purchase_type": "all",
            "num_per_page": 100,
            "cursor": cursor,
            "filter_offtopic_activity": 1,
        }

        data = None
        for attempt in range(MAX_RETRIES):
            try:
                resp = requests.get(
                    f"https://store.steampowered.com/appreviews/{appid}",
                    params=params,
                    timeout=20,
                )
                resp.raise_for_status()
                data = resp.json()
                break
            except Exception as e:
                if attempt == MAX_RETRIES - 1:
                    print(f"  ⚠️ 재시도 초과 (appid={appid}): {e}")
                    return collected
                wait = 2 ** (attempt + 1)
                print(f"  재시도 {attempt + 1}/{MAX_RETRIES} ({wait}초 대기)...", flush=True)
                time.sleep(wait)

        reviews = data.get("reviews", [])
        if not reviews:
            break

        new_cursor = data.get("cursor", "")
        if not new_cursor or new_cursor == cursor:
            break
        cursor = new_cursor

        hit_cutoff = False
        for r in reviews:
            if r.get("timestamp_created", 0) < SINCE_TS:
                hit_cutoff = True
                continue  # cutoff 이전 리뷰는 수집하지 않음

            author = r.get("author", {})
            text   = r.get("review", "")
            collected.append({
                "review_id"             : r.get("recommendationid"),
                "timestamp_created"     : r.get("timestamp_created"),
                "voted_up"              : r.get("voted_up"),
                "playtime_at_review_min": author.get("playtime_at_review"),
                "playtime_forever_min"  : author.get("playtime_forever"),
                "language"              : r.get("language"),
                "review_length"         : len(text) if text else 0,
            })

        time.sleep(SLEEP_SEC)

        if hit_cutoff:
            break  # 이 페이지에서 cutoff 이전 리뷰 발견 → 중단

    return collected


print("함수 정의 완료")

함수 정의 완료


In [4]:
if os.path.exists(OUTPUT_PATH):
    existing = pd.read_csv(OUTPUT_PATH, usecols=["app_id"])
    done_appids = set(existing["app_id"].unique())
    print(f"Resume 모드: {len(done_appids)}개 게임 이미 수집됨 → 스킵")
else:
    done_appids = set()
    print("신규 수집 시작")

total = len(TEST_GAMES)

for i, (_, game) in enumerate(TEST_GAMES.iterrows(), start=1):
    appid = int(game["appid"])
    name  = game["name"]
    genre = game["genre_category"]

    if appid in done_appids:
        print(f"[{i:02d}/{total}] {name} — 스킵")
        continue

    print(f"[{i:02d}/{total}] {name} ({appid}) 수집 중...", flush=True)
    start_time = time.time()

    reviews = collect_game_reviews(appid)

    if reviews:
        df_game = pd.DataFrame(reviews)
        df_game.insert(0, "app_id",   appid)
        df_game.insert(1, "app_name", name)
        df_game.insert(2, "genre",    genre)

        write_header = not os.path.exists(OUTPUT_PATH)
        df_game.to_csv(OUTPUT_PATH, mode="a", index=False,
                       header=write_header, encoding="utf-8-sig")

        elapsed = time.time() - start_time
        print(f"  → {len(reviews):,}개 저장 ({elapsed:.0f}초)")
    else:
        print(f"  → 리뷰 없음 또는 수집 실패")

print()
print("=" * 40)
print("테스트 수집 완료")

Resume 모드: 3개 게임 이미 수집됨 → 스킵
[01/3] Terraria — 스킵
[02/3] Stardew Valley — 스킵
[03/3] The Witcher 3 — 스킵

테스트 수집 완료


In [5]:
result = pd.read_csv(OUTPUT_PATH)

print(f"총 리뷰 수  : {len(result):,}개")
print(f"게임 수     : {result['app_id'].nunique()}개")
print()

print("=== 게임별 수집 리뷰 수 ===")
per_game = (
    result.groupby(["app_name", "genre"])["review_id"]
    .count()
    .rename("리뷰 수")
    .sort_values(ascending=False)
)
print(per_game.to_string())
print()

print("=== 결측치 확인 ===")
missing = result[["playtime_at_review_min", "playtime_forever_min", "voted_up"]].isna().sum()
missing_pct = (missing / len(result) * 100).round(1)
print(pd.DataFrame({"결측 수": missing, "결측률(%)": missing_pct}))
print()

print("=== 샘플 3행 ===")
result.head(3)

총 리뷰 수  : 900개
게임 수     : 3개

=== 게임별 수집 리뷰 수 ===
app_name        genre             
Stardew Valley  Casual/Lightweight    300
Terraria        Action                300
The Witcher 3   RPG                   300

=== 결측치 확인 ===
                        결측 수  결측률(%)
playtime_at_review_min     0     0.0
playtime_forever_min       0     0.0
voted_up                   0     0.0

=== 샘플 3행 ===


,app_id,app_name,genre,review_id,timestamp_created,voted_up,playtime_at_review_min,playtime_forever_min,language,review_length
0,105600,Terraria,Action,225850046,1779118736,True,12710,12734,russian,4
1,105600,Terraria,Action,225849863,1779118570,True,25453,25453,russian,59
2,105600,Terraria,Action,225849573,1779118321,True,3957,3967,spanish,70
